# 24 Build model-ready panel

目标：把形成月 t-1 的 59 特征严格对齐至目标月 t 的基金 rret。

- 输入：23 号同月面板、06 号 rret
- 正式输出：model_ready_panel.parquet、时间泄漏审计
- 主键：wficn+target_month
- 上游依赖：23
- smoke 输出与正式输出隔离；报告会明确标注运行模式。

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import pandas as pd
from pandas.tseries.offsets import DateOffset
from src.atomic_write import write_csv_atomic, write_parquet_atomic
from src.runtime_paths import output_path, upstream_path

names=['r2_1', 'r12_2', 'r12_7', 'r36_13', 'ST_Rev', 'LT_Rev', 'Investment', 'NOA', 'DPI2A', 'NI', 'PROF', 'ATO', 'CTO', 'FC2Y', 'OP', 'PM', 'RNA', 'ROA', 'ROE', 'SGA2S', 'D2A', 'AC', 'OA', 'OL', 'PCM', 'A2ME', 'BEME', 'C', 'CF', 'CF2P', 'D2P', 'E2P', 'Q', 'S2P', 'Lev', 'AT', 'Beta', 'IdioVol', 'LME', 'LTurnover', 'MktBeta', 'Rel2High', 'Resid_Var', 'Spread', 'SUV', 'Variance', 'F_ST_Rev', 'F_r2_1', 'F_r12_2', 'age', 'tna', 'flow', 'exp_ratio', 'turnover', 'Family_TNA', 'fund_no', 'Family_r12_2', 'Family_age', 'Family_flow']
macro_names=["Sentiment","CFNAI"]
raw=pd.read_parquet(upstream_path(ROOT,RUN_MODE,"23_build_raw_same_month_panel","outputs/panels/raw_same_month_panel.parquet"))
macro=pd.read_parquet(upstream_path(ROOT,RUN_MODE,"23_build_raw_same_month_panel","data/processed/macro/monthly_macro_features.parquet"))
y_source=pd.read_parquet(upstream_path(ROOT,RUN_MODE,"06_compute_fund_rret_and_momentum","data/processed/fund/fund_alpha_momentum_raw.parquet"))

raw["feature_month"]=pd.to_datetime(raw["month"]).dt.to_period("M").dt.to_timestamp()
raw["target_month"]=raw["feature_month"]+DateOffset(months=1)
raw["sentiment_source_month"]=raw["target_month"]-DateOffset(months=1)
raw["cfnai_source_month"]=raw["target_month"]-DateOffset(months=2)
macro["observation_month"]=pd.to_datetime(macro["observation_month"]).dt.to_period("M").dt.to_timestamp()

sentiment=macro[["observation_month","Sentiment"]].rename(
    columns={"observation_month":"sentiment_source_month"}
)
cfnai=macro[["observation_month","CFNAI"]].rename(
    columns={"observation_month":"cfnai_source_month"}
)
panel=raw[[
    "wficn","feature_month","target_month",
    "sentiment_source_month","cfnai_source_month",*names,
]].merge(
    sentiment,on="sentiment_source_month",how="left",validate="many_to_one",
).merge(
    cfnai,on="cfnai_source_month",how="left",validate="many_to_one",
)
y_source["target_month"]=pd.to_datetime(y_source["month"]).dt.to_period("M").dt.to_timestamp()
panel=panel.merge(
    y_source[["wficn","target_month","alpha"]],
    on=["wficn","target_month"],
    how="left",
    validate="one_to_one",
)
start=pd.Timestamp(PARAMS["start_target_month"])
end=pd.Timestamp(PARAMS["end_target_month"])
panel=panel.loc[(panel["target_month"]>=start)&(panel["target_month"]<=end)].copy()
panel=panel[[
    "wficn","target_month","feature_month",
    "sentiment_source_month","cfnai_source_month",
    *names,"Sentiment","CFNAI","alpha",
]].sort_values(["target_month","wficn"]).reset_index(drop=True)

assert len(names)==59 and not {"Sentiment","CFNAI","alpha","rret"} & set(names)
assert not panel.duplicated(["wficn","target_month"]).any()
assert (panel["feature_month"]==panel["target_month"]-DateOffset(months=1)).all()
assert (panel["sentiment_source_month"]==panel["target_month"]-DateOffset(months=1)).all()
assert (panel["cfnai_source_month"]==panel["target_month"]-DateOffset(months=2)).all()
assert (panel["cfnai_source_month"]<panel["feature_month"]).all()
assert "alpha" in panel.columns and "rret" not in panel.columns
assert "official_rret_raw" not in panel.columns
assert not (panel[names+macro_names+["alpha"]]==float(PARAMS["missing_export_value"])).any().any()

def one_value_per_month(frame, column):
    return frame.groupby("target_month",dropna=False)[column].nunique(dropna=False).le(1).all()
assert one_value_per_month(panel,"Sentiment")
assert one_value_per_month(panel,"CFNAI")

sentiment_missing=set(panel.loc[panel["Sentiment"].isna(),"target_month"].dt.strftime("%Y-%m").unique())
cfnai_missing=set(panel.loc[panel["CFNAI"].isna(),"target_month"].dt.strftime("%Y-%m").unique())
expected_sentiment=set()
expected_cfnai={"1980-01","1980-02"}
if RUN_MODE=="full":
    assert sentiment_missing==expected_sentiment,(sentiment_missing,expected_sentiment)
    assert cfnai_missing==expected_cfnai,(cfnai_missing,expected_cfnai)

leak=panel.loc[
    (panel["feature_month"]!=panel["target_month"]-DateOffset(months=1))
    | (panel["sentiment_source_month"]!=panel["target_month"]-DateOffset(months=1))
    | (panel["cfnai_source_month"]!=panel["target_month"]-DateOffset(months=2)),
    ["wficn","target_month","feature_month","sentiment_source_month","cfnai_source_month"],
]
macro_alignment=panel[[
    "target_month","feature_month","sentiment_source_month","cfnai_source_month",
    "Sentiment","CFNAI",
]].drop_duplicates().sort_values("target_month")
assert not macro_alignment.duplicated(["target_month"]).any()

out=output_path(ROOT,RUN_MODE,"24_build_model_ready_panel","outputs/panels/model_ready_panel.parquet")
leak_out=output_path(ROOT,RUN_MODE,"24_build_model_ready_panel","data/audit/24_time_leakage.csv")
macro_audit_out=output_path(ROOT,RUN_MODE,"24_build_model_ready_panel","data/audit/24_macro_alignment.csv")
write_parquet_atomic(panel,out)
write_csv_atomic(leak,leak_out)
write_csv_atomic(macro_alignment,macro_audit_out)
INPUTS_USED=["raw_same_month_panel.parquet","monthly_macro_features.parquet","fund_alpha_momentum_raw.parquet"]
OUTPUTS_WRITTEN=[str(path.relative_to(ROOT)) for path in [out,leak_out,macro_audit_out]]
INPUT_COUNTS={"raw_same_month":len(raw),"macro_observation_months":len(macro)}
OUTPUT_COUNTS={"model_ready":len(panel),"macro_target_months":len(macro_alignment)}
PRIMARY_KEY=["wficn","target_month"]; DUPLICATE_KEY_COUNT=0
EXCLUDED_ROW_COUNT=int(panel["alpha"].isna().sum()); EXCEPTION_FILE=str(leak_out.relative_to(ROOT))
STAGE_STATUS="passed"
STAGE_NOTES=[
    "Core features and Sentiment use target month t-1.",
    "CFNAI uses target month t-2.",
    "Expected source-coverage gaps remain NaN and no fill operation is used.",
]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "24_build_model_ready_panel", stage="24", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
